In [ ]:
import numpy as np
import pandas
import matplotlib.pyplot as plt 
import tensorflow as tf
from tensorflow.keras import Sequential, Model, layers

In [ ]:
sequences = []
dataset = pandas.read_csv(r"/kaggle/input/us-baby-names/StateNames.csv")
for i in range(300000):
    sequences.append("$" + dataset.loc[i, 'Name'] + '.')
len(sequences)

In [ ]:
sequences[-9:-1]

In [ ]:
character_set = set()
character_index = {}
max_length = max([len(name) for name in sequences])
for name in sequences:
    for ch in name:
        character_set.add(ch)
character_index = {it : i for i, it in enumerate(character_set)}
character_index[list(character_index.keys())[0]] = character_index['.']
character_index['.'] = 0
index_character = {it : i for i, it in character_index.items()}
vocab_size = len(character_index)

In [ ]:
encoder_input = np.zeros((len(sequences), max_length, len(character_index)))
decoder_input = np.zeros((len(sequences), max_length, len(character_index)))
encoder_input.shape, decoder_input.shape

In [ ]:
for name_index, name in enumerate(sequences):
    for char_index, char in enumerate(name):
        char_id = character_index[char]
        encoder_input[name_index, char_index, char_id] = 1.0
        
decoder_input[0, 0, character_index['$']] = 1.0

In [ ]:
class Seq2Seq(Model):
    def __init__(self, latent_dim = 128, vocab_size = vocab_size):
        super().__init__()
        self.encoder = Sequential([
            layers.Input(shape = (max_length, vocab_size)),
            layers.LSTM(latent_dim, return_state = True)
        ])
        self.decoder = layers.LSTM(latent_dim, return_sequences = True, return_state = True)
        self.output_layer = layers.Dense(vocab_size, activation = 'softmax')
        
    def call(self, inputs):
        encoder_input, decoder_input = inputs
        encoded, h, c = self.encoder(encoder_input)
        decoded, _, _ = self.decoder(decoder_input, initial_state = [h, c])
        out = self.output_layer(decoded)
        return out

In [ ]:
model = Seq2Seq()
model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate = 0.001), loss = 'categorical_crossentropy', metrics = ['acc'])

In [ ]:
model.fit([encoder_input, decoder_input], encoder_input, epochs = 10, batch_size = 64)

In [ ]:
y_pred = model.predict([encoder_input[:100], decoder_input[:100]])
y_pred.shape

In [ ]:
def get_tokens(predictions):
    return "".join([index_character[np.argmax(it)] if index_character[np.argmax(it)] != '.' else '' for it in predictions])
[get_tokens(y_pred[p][1:-1]) for p in range(10)]